In [3]:
import sys
import os
import requests
import pyreadstat
import pandas as pd

# 1. Resolve Project Root Directory
current_dir = os.path.abspath("")
project_root = os.path.dirname(current_dir) if os.path.basename(current_dir) == "notebooks" else current_dir
nhanes_dir = os.path.join(project_root, "data", "raw", "nhanes")
os.makedirs(nhanes_dir, exist_ok=True)

# 2. Direct Official CDC CDN URL for DEMO_G
demo_filename = "DEMO_G.XPT"
demo_path = os.path.join(nhanes_dir, demo_filename)
cdc_url = f"https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles/{demo_filename}"

# 3. Download if not exists or if file is incomplete
if not os.path.exists(demo_path) or os.path.getsize(demo_path) < 10000:
    print(f"Downloading {demo_filename} directly from CDC...")
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(cdc_url, headers=headers, stream=True, timeout=60)
    response.raise_for_status()
    with open(demo_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"Downloaded successfully: {demo_path}")

# 4. Read SAS XPT using pyreadstat
df, meta = pyreadstat.read_xport(demo_path, encoding="latin1")

# 5. Filter Adults (age >= 20)
selected_cols = ["SEQN", "RIDAGEYR", "RIAGENDR"]
adults_df = df[df["RIDAGEYR"] >= 20][selected_cols].copy()

print(f"File Size: {os.path.getsize(demo_path) / (1024*1024):.2f} MB")
print(f"Total Participants: {len(df)}")
print(f"Adults >= 20: {len(adults_df)}")
display(adults_df.head())

File Size: 3.58 MB
Total Participants: 9756
Adults >= 20: 5560


,SEQN,RIDAGEYR,RIAGENDR
0,62161.0,22.0,1.0
3,62164.0,44.0,2.0
8,62169.0,21.0,1.0
11,62172.0,43.0,2.0
13,62174.0,80.0,1.0
